In [1]:
/**
 * @todo 双爆，抗性，防御区 还没写
 */

// prettier-ignore
type RateObject = Map<string, {
    rate: number
    type: 'ATK' | 'HP' | 'DEF' | 'FIXED'
    tags: string[]
}>

interface BuffDelta {
    atk_fixed: number // 固定攻击力
    atk_percent: number // 百分比攻击力
    hp_fixed: number // 固定生命值
    hp_percent: number // 百分比生命值
    def_fixed: number // 固定防御
    def_percent: number // 百分比防御
    inc_general: number // 通用属性增伤
    amp_general: number // 通用属性加深
    incs: Map<string, number> // 各类型增伤区
    amps: Map<string, number> // 各类型加深区
    tune_break_boost: number // 谐度破坏增幅
    tune_strain_stack: number // 集谐层数
    totals: Map<string, number> // 各终伤区
}

interface Buff {
    name: string
    delta: BuffDelta
}

type AxisObject = Array<{
    atk: number
    hp: number
    def: number
    incs: Map<string, number>
    amps: Map<string, number>
    totals: Map<string, number>
    source: {
        charactor: string
        skill: string
    }
}>

const safeCalculate = (expr: string) => {
    try {
        return parseFloat(Function(`'use strict';return(${expr})`)())
    } catch {
        return 0
    }
}

const two_sum = (a: number, b: number) => a + b

function toRateObject(charactorRateText: string): RateObject {
    const result: RateObject = new Map()

    const skills = charactorRateText
        .split('\n')
        .map((s) => s.split('//')[0])
        .filter((s) => !!s.trim())
        .map((s) => s.trim())

    skills
        .map((skill) => skill.split(/ +/))
        .forEach(([name, rate, ...tags]) => {
            if (!rate) return

            const fixed = rate.includes('fixed')

            result.set(name, {
                rate: safeCalculate(rate) / (fixed ? 1 : 100),
                type:
                    rate.includes('HP') ? 'HP'
                    : rate.includes('DEF') ? 'DEF'
                    : fixed ? 'FIXED'
                    : 'ATK',
                tags: tags as any,
            })
        })

    return result
}

function toAxisObject(axisText: string): AxisObject {
    const result: AxisObject = []

    const charactors: string[] = []
    const base_atks: number[] = []
    const base_hps: number[] = []
    const base_defs: number[] = []

    const buff_map = new Map<string, Buff>()
    const current_buffs: Buff[][] = [[], [], []]

    const lines = axisText
        .split('\n')
        .map((s) => s.split('//')[0])
        .filter((s) => !!s.trim())
        .map((s) => s.trim())

    lines
        .map((skill) => skill.split(/ +/))
        .forEach(([operation, ...args]) => {
            if (operation.startsWith('@charactor')) {
                const [name, ...bases] = args
                let index = charactors.findIndex((v) => v === name)
                index = index === -1 ? charactors.length : index
                charactors[index] = name
                base_atks[index] = safeCalculate((bases.find((v) => v.startsWith('base_atk=')) ?? '=0').split('=')[1]) // prettier-ignore
                base_hps[index] = safeCalculate((bases.find((v) => v.startsWith('base_hp=')) ?? '=0').split('=')[1]) // prettier-ignore
                base_defs[index] = safeCalculate((bases.find((v) => v.startsWith('base_def=')) ?? '=0').split('=')[1]) // prettier-ignore
                return
            }

            if (operation.startsWith('@declare_buff')) {
                const [name, ...effects] = args
                const delta = {} as BuffDelta

                delta.incs = new Map()
                delta.amps = new Map()
                delta.totals = new Map()

                // prettier-ignore
                for (const part of ['atk_fixed', 'atk_percent', 'hp_fixed', 'hp_percent', 'def_fixed', 'def_percent', 'inc_general', 'amp_general', 'tune_break_boost', 'tune_strain_stack'] as const) {
                    delta[part] = safeCalculate((effects.find((v) => v.startsWith(`${part}+=`)) ?? '+=0').split('+=')[1]) // prettier-ignore
                }

                for (const prefix of ['inc', 'amp', 'total'] as const) {
                    const regex = new RegExp(
                        `^${prefix}\\((.*?)\\)\\+\\=(.*)$`,
                        'i',
                    )
                    effects
                        .filter((effect) => regex.test(effect))
                        .forEach((effect) => {
                            const match = effect.match(regex)
                            if (!match) return
                            const [, name, expr] = match
                            delta[`${prefix}s`].set(`${prefix}(${name})`, safeCalculate(expr)) // prettier-ignore
                        })
                }

                buff_map.set(name, { name, delta }) // prettier-ignore
                return
            }

            if (operation.startsWith('@apply_buff_on')) {
                const [charactor_names, buff_name] = args
                charactor_names.split(',').forEach((charactor) => {
                    const index = charactors.findIndex((c) => c === charactor)
                    if (index === -1 || !buff_map.has(buff_name)) return

                    current_buffs[index].push(buff_map.get(buff_name)!)
                })
                return
            }

            if (operation.startsWith('@remove_buff_from')) {
                const [charactor_names, buff_name] = args
                charactor_names.split(',').forEach((charactor) => {
                    const index = charactors.findIndex((c) => c === charactor)
                    if (index === -1 || !buff_map.has(buff_name)) return

                    const buff_index = current_buffs[index].findLastIndex(b => b.name === buff_name) // prettier-ignore
                    if (index !== -1) current_buffs.splice(buff_index, 1)
                    return
                })
            }

            const [charactor, skill] = operation.split('.')

            const index = charactors.findIndex((c) => c === charactor)
            if (index === -1) return

            const current_buff = current_buffs[index].map((buff) => buff.delta)
            result.push({
                // prettier-ignore
                atk: (
                    base_atks[index] *
                        current_buff
                            .map((d) => d.atk_percent)
                            .reduce(two_sum, 100)) / 100
                    + current_buff
                        .map((d) => d.atk_fixed)
                        .reduce(two_sum, 0
                ),
                // prettier-ignore
                hp: (
                    base_hps[index] *
                        current_buff
                            .map((d) => d.hp_percent)
                            .reduce(two_sum, 100)
                ) / 100 
                    + current_buff
                        .map((d) => d.hp_fixed)
                        .reduce(two_sum, 0),
                // prettier-ignore
                def: (
                    base_defs[index] *
                        current_buff
                            .map((d) => d.def_percent)
                            .reduce(two_sum, 100)
                ) / 100
                    + current_buff
                        .map((d) => d.def_fixed)
                        .reduce(two_sum, 0),
                // prettier-ignore 增伤区
                incs: new Map<string, number>(
                    Object.freeze([
                            ...current_buff.flatMap(d => [...d.incs.entries()])
                                .reduce((m, [k, v]) => m.set(k, (m.get(k) || 0) + v), new Map<string, number>())
                                .entries()
                        ].map(
                            ([k, v]) => [k, (v + 100) / 100] as [string, number]
                        )
                    )
                ).set('__general__', current_buff.map(d => d.inc_general).reduce(two_sum, 0) / 100),
                // prettier-ignore 加深区
                amps: new Map<string, number>(
                    Object.freeze([
                            ...current_buff.flatMap(d => [...d.amps.entries()])
                                .reduce((m, [k, v]) => m.set(k, (m.get(k) || 0) + v), new Map<string, number>())
                                .entries()
                        ].map(
                            ([k, v]) => [k, (v + 100) / 100] as [string, number]
                        )
                    )
                ).set('__general__', current_buff.map(d => d.amp_general).reduce(two_sum, 0) / 100),
                // prettier-ignore 终伤区和集谐
                totals: new Map<string, number>(
                    Object.freeze([
                            ...current_buff.flatMap(d => [...d.totals.entries()])
                                .reduce((m, [k, v]) => m.set(k, (m.get(k) || 0) + v), new Map<string, number>())
                                .entries()
                        ].map(
                            ([k, v]) => [k, (v + 100) / 100] as [string, number]
                        )
                    )
                ).set('__tune_strain__', 1 + 0.0012
                    * current_buff.map(d => d.tune_break_boost).reduce(two_sum, 0) 
                    * current_buff.map(d => d.tune_strain_stack).reduce(two_sum, 0)
                ),
                source: { charactor, skill },
            })
        })

    return result
}

function calcFinalDamage(
    charactors: string[],
    rateTexts: string[],
    axisText: string,
) {
    const axisObject = toAxisObject(axisText)
    const rateObjects = rateTexts.map((t) => toRateObject(t))
    let result = 0

    for (const operation of axisObject) {
        const index = charactors.findIndex(
            (c) => c === operation.source.charactor,
        )
        if (index === -1) {
            throw new Error(`未知角色 ${operation.source.charactor}`)
        }

        const rateObject = rateObjects[index]
        if (!rateObject.has(operation.source.skill)) {
            throw new Error(`未定义技能 ${operation.source.skill}`)
        }

        const { rate, type, tags } = rateObject.get(operation.source.skill)!

        if (type === 'FIXED') {
            result += rate
            continue
        }

        // @ts-ignore
        const base: number = operation[type.toLowerCase()]
        // prettier-ignore
        const inc = 1
            + (operation.incs.get('__general__') ?? 0) 
            + tags.map(t => operation.incs.get(t) ?? 0).reduce(two_sum)
        // prettier-ignore
        const amp = 1 
            + (operation.amps.get('__general__') ?? 0) 
            + tags.map(t => operation.incs.get(t) ?? 0).reduce(two_sum)
        const total = operation.totals.values().reduce((a, b) => a * b)

        result += base * inc * amp * rate * total
    }

    // console.log(axisObject, rateObjects)
    return result
}


In [2]:
const denia_rate = `
// 固定伤害
burst        32096_FIXED           其它 // 矩阵R1平抗10层聚爆伤害（暂时用不到）

// 常态攻击
a1_w         32.69                 普攻 // 白娅 地面 a1 a2 a3 a4
a2_w         30.18*2               普攻
a3_w         25.49*3               普攻
a4_w         128.00                普攻

z_w          80.76*2               重击 // 白娅重击（不打）
x_w          29.59+44.38           普攻 // 白娅下落攻击（不打）
s_w          49.35*3               普攻 // 白娅闪避反击（不打）

a1_b_gnd     36.51                 普攻 // 黑娅 地面 a1 a2 a3 a4（无虚质粒子、由本行起）
a2_b_gnd     37.51+14.07*4         普攻
a3_b_gnd     62.39                 普攻
a4_b_gnd     35.54+82.92           普攻

z_b_gnd      137.06                重击 // 黑娅重击（不打）
z_b_sky      29.59+44.38           重击 // 黑娅空中重击（不打）
s_b          108.08                普攻 // 黑娅闪避反击（不打）

a1_b_sky     36.51                 普攻 // 黑娅 空中 a1 a2 a3 a4
a2_b_sky     37.51+14.07*4         普攻    
a3_b_sky     62.39                 普攻
a4_b_sky     35.54+82.92           普攻

a1_b_gnd!    1.5*(36.51)           解放 // 黑娅 地面 a1 a2 a3 a4（有虚质粒子、由本行起）
a2_b_gnd!    1.5*(37.51+14.07*4)   解放
a3_b_gnd!    1.5*(62.39)           解放
a4_b_gnd!    1.5*(35.54+82.92)     解放

z_b_gnd!     1.5*(137.06)          解放 // 黑娅重击（不打）
z_b_sky!     1.5*(29.59+44.38)     解放 // 黑娅空中重击（不打）
s_b!         1.5*(108.08)          解放 // 黑娅闪避反击（不打）

a1_b_sky!    1.5*(36.51)           解放 // 黑娅 空中 a1 a2 a3 a4
a2_b_sky!    1.5*(37.51+14.07*4)   解放    
a3_b_sky!    1.5*(62.39)           解放
a4_b_sky!    1.5*(35.54+82.92)     解放

// 共鸣技能
e_w          17.42*3+52.25         共技
e1_b         31.10+24.52*5         共技
e2_b_1       34.68*3               解放
e2_b_2       112.01                解放

// 共鸣解放
r1           397.62                解放
r2           198.81*4              解放

// 变奏技能
intro_w      104.62                变奏 // 注意：白娅变奏派生a4
intro_b      51.74*3               变奏

// 共鸣回路
sy           136.33                解放 // 蚀域每次伤害

// 声骸技能
echo         82.08+191.52          声骸
`

In [3]:
const preparation = `
@charactor denia base_atk=945
@charactor lupa base_atk=974

@declare_buff 熵变强化-幻灭之形 atk_percent+=30
@declare_buff 蚀刻繁彩-聚爆模态下-熵变强化时-全队-热熔 inc_general+=30
@declare_buff 蚀刻繁彩-集谐模态下-熵变强化时-莫宁谐振场内-全队 
@declare_buff 集谐响应后 tune_break_boost+=40
@declare_buff 集谐户口 tune_strain_stack+=1
@declare_buff 达妮娅延奏-集谐模态下-下一位角色 amp_general+=15
@declare_buff 达妮娅延奏-集谐模态下-下一位角色-响应集谐后 amp_general+=-15+40

@declare_buff 露帕追猎-攻击力-开大/团队变奏时-前台 atk_percent+=6
@declare_buff 露帕追猎-增伤-前台-热熔 inc_general+=10
@declare_buff 露帕追猎-增伤-三火队-前台-热熔 inc_general+=20
@declare_buff 露帕-所处皆为猎场-1-全队-热熔 inc_general+=20
@declare_buff 露帕-所处皆为猎场-2-全队-热熔 inc_general+=20
@declare_buff 露帕-专武1阶-全队-热熔 inc_general+=24

@declare_buff 达尼娅-专武1阶-自身-常驻 atk_percent+=12
@declare_buff 达尼娅-专武1阶-自身-附加聚爆效应/集谐偏移时 inc(解放)+=36
@declare_buff 达尼娅-专武1阶-特定角色-附加聚爆效应/集谐偏移时 atk_percent+=24

@declare_buff 火套二件套通用 inc_general+=10
@declare_buff 奔狼燎原之焰-首位荣耀狮像 inc_general+=12 inc(解放)+=12
@declare_buff 奔狼燎原之焰-施放共鸣解放-全队-热熔 inc_general+=15
@declare_buff 奔狼燎原之焰-施放共鸣解放-自身-解放 inc(解放)+=20
@declare_buff 斑驳粉饰之沫-添加聚爆效应-自身-热熔 inc_general+=10
@declare_buff 斑驳粉饰之沫-添加聚爆效应延奏下一位-首位达尼亚共鸣回响-热熔 inc_general+=12+25

// 注意：添加聚爆效应手段为任意a3a4，r1r2，变奏，蚀域
`

In [4]:
const atk_atk_3c = `
@declare_buff 装备声骸 atk_fixed+=150+100+100 atk_percent+=30+30+18+18 inc_general+=0 inc(解放)+=0
`

const atk_fusion_3c = `
@declare_buff 装备声骸 atk_fixed+=150+100+100 atk_percent+=30+18+18 inc_general+=32 inc(解放)+=0
`

const fusion_fusion_3c = `
@declare_buff 装备声骸 atk_fixed+=150+100+100 atk_percent+=18+18 inc_general+=32+32 inc(解放)+=0
`

const sub_attr = `
@declare_buff 声骸副词条 atk_fixed+=50+40 atk_percent+=10.9+8.6 inc(解放)+=24.4 inc(普攻)+=8.6 inc(共技)+=8.6
@apply_buff_on denia 声骸副词条
`

In [ ]:
const main_axis = `
@apply_buff_on denia 装备声骸
denia.e_w

@apply_buff_on denia,lupa 蚀刻繁彩-聚爆模态下-熵变强化时-全队-热熔
@apply_buff_on denia 达尼娅-专武1阶-自身-常驻
@apply_buff_on denia 达尼娅-专武1阶-自身-附加聚爆效应/集谐偏移时
@apply_buff_on denia 达尼娅-专武1阶-特定角色-附加聚爆效应/集谐偏移时
@apply_buff_on denia 火套二件套通用
@apply_buff_on denia 奔狼燎原之焰-首位荣耀狮像
@apply_buff_on denia,lupa 奔狼燎原之焰-施放共鸣解放-全队-热熔
@apply_buff_on denia 奔狼燎原之焰-施放共鸣解放-自身-解放
denia.r1

denia.e1_b
denia.echo

denia.a1_b_sky!
denia.a2_b_sky!

@apply_buff_on lupa 露帕追猎-攻击力-开大/团队变奏时-前台
@apply_buff_on lupa 露帕追猎-增伤-三火队-前台-热熔
@apply_buff_on denia,lupa 露帕-专武1阶-全队-热熔
@apply_buff_on lupa 奔狼燎原之焰-施放共鸣解放-自身-解放
@apply_buff_on denia,lupa 奔狼燎原之焰-施放共鸣解放-全队-热熔
@apply_buff_on denia,lupa 露帕-所处皆为猎场-1-全队-热熔 // c2
// lupa.r // 阻止报错

@remove_buff_from lupa 露帕追猎-攻击力-开大/团队变奏时-前台
@remove_buff_from lupa 露帕追猎-增伤-三火队-前台-热熔
@apply_buff_on denia 露帕追猎-攻击力-开大/团队变奏时-前台
@apply_buff_on denia 露帕追猎-增伤-三火队-前台-热熔

denia.a3_b_sky!
denia.a4_b_sky!
// lupa.A1 // 阻止报错
@apply_buff_on denia 露帕-所处皆为猎场-2-全队-热熔 // c2

denia.e2_b_1
denia.e2_b_2

denia.r2
@remove_buff_from denia 熵变强化-幻灭之形

denia.sy
denia.sy
denia.sy
denia.sy
denia.sy
denia.sy
denia.sy
denia.sy
`

In [6]:
console.log(
    '双攻击',
    calcFinalDamage(
        ['denia'],
        [denia_rate],
        preparation + atk_atk_3c + main_axis,
    ),
    '一攻一属',
    calcFinalDamage(
        ['denia'],
        [denia_rate],
        preparation + atk_fusion_3c + main_axis,
    ),
    '双属',
    calcFinalDamage(
        ['denia'],
        [denia_rate],
        preparation + fusion_fusion_3c + main_axis,
    ),
)


双攻击 212179.65490890003 一攻一属 214478.46067899995 双属 210430.34102510003


In [7]:
// 露帕 2 链 首轮无加深 一攻一属优
(214478.46 - 210430.34) / 210430.34


0.019237340014752604

In [8]:
// 露帕 0 链 首轮无加深 一攻一属优
(193753.21 - 192242.50) / 192242.50

0.007858355982678086